In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
data = pd.read_csv("used_cars.csv")

print("Dataset Shape:", data.shape)

data.head()

Dataset Shape: (4009, 12)


,brand,model,model_year,milage,fuel_type,engine,transmission,ext_col,int_col,accident,clean_title,price
0,Ford,Utility Police Interceptor Base,2013,"51,000 mi.",E85 Flex Fuel,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,6-Speed A/T,Black,Black,At least 1 accident or damage reported,Yes,"$10,300"
1,Hyundai,Palisade SEL,2021,"34,742 mi.",Gasoline,3.8L V6 24V GDI DOHC,8-Speed Automatic,Moonlight Cloud,Gray,At least 1 accident or damage reported,Yes,"$38,005"
2,Lexus,RX 350 RX 350,2022,"22,372 mi.",Gasoline,3.5 Liter DOHC,Automatic,Blue,Black,None reported,NaN,"$54,598"
3,INFINITI,Q50 Hybrid Sport,2015,"88,900 mi.",Hybrid,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,7-Speed A/T,Black,Black,None reported,Yes,"$15,500"
4,Audi,Q3 45 S line Premium Plus,2021,"9,835 mi.",Gasoline,2.0L I4 16V GDI DOHC Turbo,8-Speed Automatic,Glacier White Metallic,Black,None reported,NaN,"$34,999"


In [3]:
print(data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4009 entries, 0 to 4008
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   brand         4009 non-null   object
 1   model         4009 non-null   object
 2   model_year    4009 non-null   int64 
 3   milage        4009 non-null   object
 4   fuel_type     3839 non-null   object
 5   engine        4009 non-null   object
 6   transmission  4009 non-null   object
 7   ext_col       4009 non-null   object
 8   int_col       4009 non-null   object
 9   accident      3896 non-null   object
 10  clean_title   3413 non-null   object
 11  price         4009 non-null   object
dtypes: int64(1), object(11)
memory usage: 376.0+ KB
None


In [4]:
print("Missing Values:")
print(data.isnull().sum())

Missing Values:
brand             0
model             0
model_year        0
milage            0
fuel_type       170
engine            0
transmission      0
ext_col           0
int_col           0
accident        113
clean_title     596
price             0
dtype: int64


In [5]:
print("Duplicate Rows:", data.duplicated().sum())

Duplicate Rows: 0


In [6]:
data["price"] = (
    data["price"]
    .astype(str)
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

data["price"] = pd.to_numeric(data["price"], errors="coerce")

In [7]:
data["milage"] = (
    data["milage"]
    .astype(str)
    .str.replace("mi.", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

data["milage"] = pd.to_numeric(data["milage"], errors="coerce")

In [8]:
data[["milage", "price"]].head()

,milage,price
0,51000,10300
1,34742,38005
2,22372,54598
3,88900,15500
4,9835,34999


In [9]:
data = data.dropna(subset=["price"])

print("Dataset Shape After Cleaning:", data.shape)

Dataset Shape After Cleaning: (4009, 12)


In [10]:
CURRENT_YEAR = 2026

data["car_age"] = CURRENT_YEAR - data["model_year"]

data["car_age"] = data["car_age"].clip(lower=0)

data[["model_year", "car_age"]].head()

,model_year,car_age
0,2013,13
1,2021,5
2,2022,4
3,2015,11
4,2021,5


In [11]:
data["horsepower"] = (
    data["engine"]
    .astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)HP", expand=False)
)

data["horsepower"] = pd.to_numeric(
    data["horsepower"],
    errors="coerce"
)

In [12]:
data["engine_liters"] = (
    data["engine"]
    .astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)L", expand=False)
)

data["engine_liters"] = pd.to_numeric(
    data["engine_liters"],
    errors="coerce"
)

In [13]:
data[
    ["engine", "horsepower", "engine_liters"]
].head(10)

,engine,horsepower,engine_liters
0,300.0HP 3.7L V6 Cylinder Engine Flex Fuel Capa...,300.0,3.7
1,3.8L V6 24V GDI DOHC,NaN,3.8
2,3.5 Liter DOHC,NaN,NaN
3,354.0HP 3.5L V6 Cylinder Engine Gas/Electric H...,354.0,3.5
4,2.0L I4 16V GDI DOHC Turbo,NaN,2.0
5,2.4 Liter,NaN,NaN
6,292.0HP 2.0L 4 Cylinder Engine Gasoline Fuel,292.0,2.0
7,282.0HP 4.4L 8 Cylinder Engine Gasoline Fuel,282.0,4.4
8,311.0HP 3.5L V6 Cylinder Engine Gasoline Fuel,311.0,3.5
9,534.0HP Electric Motor Electric Fuel System,534.0,NaN


In [14]:
data["has_accident"] = (
    data["accident"]
    .fillna("Unknown")
    .astype(str)
    .str.contains("accident", case=False)
    .astype(int)
)

In [15]:
data["clean_title_flag"] = (
    data["clean_title"]
    .fillna("Unknown")
    .astype(str)
    .str.lower()
    .map({
        "yes": 1,
        "no": 0
    })
)

data["clean_title_flag"] = data["clean_title_flag"].fillna(0)

In [16]:
upper_price = data["price"].quantile(0.99)

print("99th Percentile Price:", upper_price)

data = data[data["price"] <= upper_price].copy()

print("Shape after removing extreme prices:", data.shape)

99th Percentile Price: 272713.2800000002
Shape after removing extreme prices: (3968, 17)


In [17]:
features = [
    "brand",
    "model",
    "model_year",
    "car_age",
    "milage",
    "fuel_type",
    "engine",
    "horsepower",
    "engine_liters",
    "transmission",
    "ext_col",
    "int_col",
    "accident",
    "clean_title",
    "has_accident",
    "clean_title_flag"
]

X = data[features]
y = data["price"]

print("X Shape:", X.shape)
print("y Shape:", y.shape)

X Shape: (3968, 16)
y Shape: (3968,)


In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

Training: (3174, 16)
Testing : (794, 16)


In [19]:
numeric_features = [
    "model_year",
    "car_age",
    "milage",
    "horsepower",
    "engine_liters",
    "has_accident",
    "clean_title_flag"
]

categorical_features = [
    "brand",
    "model",
    "fuel_type",
    "engine",
    "transmission",
    "ext_col",
    "int_col",
    "accident",
    "clean_title"
]

In [20]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [21]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [22]:
rf_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "model",
            RandomForestRegressor(
                n_estimators=300,
                max_depth=20,
                min_samples_split=5,
                min_samples_leaf=2,
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

In [23]:
rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

In [24]:
rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest Performance")
print("-------------------------")
print(f"MAE  : ${rf_mae:,.2f}")
print(f"RMSE : ${rf_rmse:,.2f}")
print(f"R²   : {rf_r2:.4f}")
print(f"R² % : {rf_r2:.2%}")

Random Forest Performance
-------------------------
MAE  : $8,823.70
RMSE : $17,685.25
R²   : 0.7342
R² % : 73.42%


In [28]:
new_car = pd.DataFrame({
    "brand": ["Toyota"],
    "model": ["Corolla LE"],
    "model_year": [2020],
    "car_age": [6],
    "milage": [50000],
    "fuel_type": ["Gasoline"],
    "engine": ["1.8L 4 Cylinder Engine"],
    "horsepower": [None],
    "engine_liters": [1.8],
    "transmission": ["Automatic"],
    "ext_col": ["White"],
    "int_col": ["Black"],
    "accident": ["None reported"],
    "clean_title": ["Yes"],
    "has_accident": [0],
    "clean_title_flag": [1]
})

In [29]:
if gb_r2 > rf_r2:
    final_model = gb_model
else:
    final_model = rf_model

In [30]:
predicted_price = final_model.predict(new_car)

print(
    f"Predicted Used Car Price: "
    f"${predicted_price[0]:,.2f}"
)

Predicted Used Car Price: $32,954.30
